
Il problema del Set Cover è **NP-difficile**, quindi la soluzione ottimale può richiedere tempi computazionalmente elevati. Per questo motivo in questo laboratorio:
- Dovremo implementare e confrontare vari algoritmi di ricerca locale come Hill Climbing, Simulated Annealing, e Tabu Search per affrontare il problema del Set Cover.
- Per ogni algoritmo e per ogni istanza del problema (variazioni delle dimensioni dell'universo, numero dei set, e densità), dovremo restituire un **costo finale** e il **numero di passi** compiuti per trovare la soluzione.

Questo approccio ci permette di valutare l'efficienza degli algoritmi su varie dimensioni e configurazioni del problema.



1. **Input**:
   - **Universo degli elementi**: un insieme \( U \) di elementi, indicato con una cardinalità \( |U| \). Ad esempio, per un universo di dimensione 5, potremmo avere \( U = \{1, 2, 3, 4, 5\} \).
   - **Insieme di insiemi**: una collezione \( S = \{S_1, S_2, \dots, S_n\} \) di sottoinsiemi di \( U \). Ogni \( S_i \) rappresenta un sottoinsieme di \( U \), e possiamo immaginare che copra una parte dell'universo.
   - **Costo dei set**: ogni sottoinsieme \( S_i \) ha un costo associato, indicato come \( c(S_i) \).

2. **Obiettivo**:
   - Trovare un sottoinsieme minimo di \( S \) (un **coprente minimo**) tale che l’unione dei set selezionati copra l'intero universo \( U \).
   - Minimizzare il costo totale della selezione.



Immagina che:
- L’universo degli elementi sia \( U = \{1, 2, 3, 4, 5\} \).
- Ci siano i seguenti set:
  - \( S_1 = \{1, 2\} \) con costo 1.
  - \( S_2 = \{2, 3, 4\} \) con costo 2.
  - \( S_3 = \{4, 5\} \) con costo 1.
- La soluzione minima sarebbe quella che copre tutto \( U \) spendendo il meno possibile.


- **`numpy`** ci permette di gestire matrici e array in modo efficiente, rendendo le operazioni matematiche e logiche su questi tipi di dati veloci e ottimizzate.
- **`functools`** con `lru_cache` aiuta a evitare ricalcoli inutili, ottimizzando il tempo di esecuzione.
- **`tqdm`** è utile per visualizzare una barra di progresso, particolarmente pratica per algoritmi che richiedono molti passi.
- **`icecream`** con `ic` ci fornisce stampe dettagliate per il debug, facilitando il controllo dei valori intermedi e il monitoraggio dello stato del programma.
- **`rng`** garantisce che i numeri casuali siano riproducibili, importante per confrontare e replicare i risultati tra esecuzioni.


In [253]:
import functools  
from itertools import accumulate
from datetime import timedelta
import numpy as np 
from tqdm.auto import tqdm  
from icecream import ic
import matplotlib.pyplot as plt
import time
import random


Inizializzazione variabili globali UNIVERSE_SIZE, NUM_SETS, DENSITY, e rng per memorizzare i parametri dell'istanza corrente.
- **`UNIVERSE_SIZE`**: rappresenta la dimensione dell'universo, cioè il numero di elementi da coprire.
- **`NUM_SETS`**: indica quanti set useremo per coprire l'universo.
- **`DENSITY`**: probabilità che un elemento dell'universo sia contenuto in un set.
- **`rng`**: inizializzato con `PCG64` di `numpy`, fornisce un generatore di numeri casuali che dipende dai parametri della nostra istanza. Questo permette di ottenere risultati riproducibili per ogni configurazione.

In [254]:
UNIVERSE_SIZE = None
NUM_SETS = None
DENSITY = None
rng = None

SETS = None
COSTS = None

La funzione `generate_universe` imposta i parametri globali per l'istanza corrente e crea un generatore di numeri casuali specifico per l'istanza.
  - Inizializza il generatore `rng` usando `np.random.PCG64` con un seed specifico, basato sui parametri dell'istanza (universe_size, num_sets, density), questo assicura che ogni configurazione generi numeri casuali riproducibili.


In [255]:
def generate_universe(universe_size, num_sets, density):
    """
    Configures global parameters for the universe and initializes the random generator.

    Parameters:
        universe_size (int): Size of the universe (number of elements).
        num_sets (int): Number of sets available.
        density (float): Density of coverage for each set.
    """
    global UNIVERSE_SIZE, NUM_SETS, DENSITY, rng
    
    UNIVERSE_SIZE = universe_size
    NUM_SETS = num_sets
    DENSITY = density
    
    # Inizializzazione del generatore di numeri casuali specifico per l'istanza
    seed = UNIVERSE_SIZE * 1000000 + NUM_SETS * 1000 + int(DENSITY * 1000)
    rng = np.random.Generator(np.random.PCG64(seed))
    ic(f"Generator initialized for Universe Size: {UNIVERSE_SIZE}, Num Sets: {NUM_SETS}, Density: {DENSITY}")

La funzione `generate_instance` crea l'istanza specifica del problema del Set Cover, composta dalla matrice `SETS` e dall’array `COSTS`.

- **Passaggi**:
  1. **Generazione della matrice `SETS`**:
     - Ogni cella SETS[i, j] viene inizializzata in modo probabilistico in base alla densità. Questo può lasciare alcuni elementi dell'universo non coperti da alcun set.
  2. **Verifica della Copertura Completa**:
     - La funzione scorre ogni elemento dell'universo (for s in range(UNIVERSE_SIZE)) per verificare se è coperto. Se l'elemento s non è coperto da nessun set (ossia, SETS[:, s] è tutto False), allora il programma assegna True a un set casuale per quell’elemento.
  3. **Calcolo dei Costi (`COSTS`)**:
     - Il costo di ogni set è proporzionale alla somma degli elementi coperti, con una penalizzazione per i set più grandi. `np.power(SETS.sum(axis=1), 1.1)` applica una penalità per ridurre il bias verso i set che coprono troppi elementi.

In [256]:
def generate_instance():
    """
    Generates a problem instance for the Set Cover problem with predefined settings.

    Returns:
        SETS (np.array): Boolean matrix indicating the coverage of each element by each set.
        COSTS (np.array): Array of costs for each set.
    """
    global SETS, COSTS
    
    # Creazione dei set con copertura probabilistica
    SETS = rng.random((NUM_SETS, UNIVERSE_SIZE)) < DENSITY
    
    # Assicurazione della copertura completa per ogni elemento dell'universo
    for s in range(UNIVERSE_SIZE):
        if not np.any(SETS[:, s]):
            selected_set = rng.integers(NUM_SETS)
            SETS[selected_set, s] = True  # Assegna l'elemento a un set casuale
    
    # Calcolo dei costi con penalizzazione per i set più grandi
    COSTS = np.power(SETS.sum(axis=1), 1.1)



Usiamo il codice seguente per creare automaticamente tutte le istanze definite nella lista `instances`. Questo ciclo permette di configurare i parametri, generare `SETS` e `COSTS`, e stamparli per ogni configurazione.



La funzione `tweak` genera una **soluzione candidata** modificando un set casuale della soluzione corrente:
- **Input**: la soluzione corrente (lista o array di valori booleani).
- **Output**: una nuova soluzione che rappresenta una piccola variazione della soluzione originale.

Il cambiamento deve essere minimo, per mantenere la soluzione vicina a quella attuale e ottenere un "vicinato" della soluzione iniziale.


In [257]:
def tweak(solution, elements):
    """
    Generates a candidate solution by modifying a random set's inclusion and updates the elements coverage.

    Parameters:
        solution (np.array): Current solution representing selected sets.
        elements (np.array): Current coverage array representing the count of each element being covered.

    Returns:
        tuple: (candidate (np.array), updated_elements (np.array))
            - candidate: A slightly modified solution.
            - updated_elements: Updated coverage array after tweaking.
    """
    candidate = solution.copy()
    index = rng.integers(0, NUM_SETS)
    candidate[index] = not candidate[index]
    diff = 2 * candidate[index] - 1

    # Evita soluzioni invalide
    if diff == -1 and np.any(elements[SETS[index]] <= 1):
        return solution, elements

    elements_updated = elements + diff * SETS[index]
    return candidate, elements_updated



La funzione `cost` valuta una soluzione calcolandone il costo totale:
- **Input**: `solution`, una lista o array booleano che indica quali set sono selezionati.
- **Output**: `cost`, il costo totale dei set selezionati, sommando i rispettivi valori in `COSTS`.

Una soluzione con un costo inferiore è preferibile, poiché indica che copre l'universo con meno risorse (o costo) possibile.


In [258]:
def cost(solution):
    """
    Calculates the total cost of the selected solution.

    Parameters:
        solution (np.array): Boolean array indicating selected sets (True for included sets).

    Returns:
        float: Total cost of the selected sets.
    """
    total_cost = COSTS[solution].sum()
    return total_cost



La funzione `valid` controlla se una soluzione candidata fornisce una copertura completa dell’universo:
- **Input**: `solution`, una lista o array booleano che indica i set selezionati.
- **Output**: `True` se la soluzione copre tutti gli elementi dell'universo, `False` altrimenti.

1. **Calcolo della Copertura**:
   - `SETS[solution]` seleziona i set inclusi nella soluzione.
   - `np.any(SETS[solution], axis=0)` calcola un vettore di copertura, dove ogni valore indica se l’elemento corrispondente dell'universo è coperto almeno una volta.

2. **Verifica della Copertura Completa**:
   - `np.all(coverage)` controlla se tutti i valori del vettore di copertura sono `True`, indicativo di una copertura completa dell’universo.

In [259]:
def valid(elements):
    """
    Checks whether a solution is valid (i.e., covers the entire universe).

    Parameters:
        elements (np.array): Array indicating coverage of each element.

    Returns:
        bool: True if the solution covers the entire universe, False otherwise.
    """
    is_valid = np.all(elements > 0)
    return is_valid



La funzione `fitness` valuta quanto bene una soluzione copre l'universo e ne misura il costo. È utile per analizzare la qualità di una soluzione in termini di:

- **Input**:
  - `solution`: una lista o array booleano che rappresenta i set selezionati (True per i set inclusi, False per quelli esclusi).
  - `elements`: un array che indica gli elementi dell'universo da coprire.

- **Output**:
  - Un **tupla** con `(coverage_score, -solution_cost)`, dove:
    - `coverage_score`: numero di elementi dell'universo coperti dalla soluzione (valore positivo).
    - `solution_cost`: costo totale dei set selezionati (valore negativo, per indicare che un costo più basso è preferibile).

In [260]:
def fitness(solution, elements):
    """
    Calculates the fitness of a solution based on coverage and cost.

    Parameters:
        solution (np.array): Boolean array representing selected sets.
        elements (np.array): Array indicating coverage of each element.

    Returns:
        tuple: (coverage_score, -cost), where:
            - coverage_score: Number of elements covered by the solution.
            - cost: Negative of the total cost of the solution (to minimize).
    """
    coverage_score = np.sum(elements > 0)
    solution_cost = cost(solution)

    return (coverage_score, -solution_cost)





L'algoritmo di Hill Climbing è una tecnica di **ricerca locale**:

1. **Inizializzazione**: la soluzione corrente e il miglior costo vengono impostati sulla soluzione iniziale e il suo costo.
2. **Ciclo di Ottimizzazione**:
   - Per ciascun passo, `tweak` genera una soluzione candidata vicina.
   - Se la soluzione è valida e ha un costo inferiore, viene accettata come nuova soluzione corrente.
   - Se questa soluzione migliora il miglior costo, viene aggiornata come la miglior soluzione trovata.
3. **Restituzione della Miglior Soluzione**: al termine del ciclo, la funzione restituisce la miglior soluzione e il costo associato.

In [261]:
def hill_climbing(max_steps):
    """
    Algoritmo di Hill Climbing per il problema del Set Cover.
    """
    solution = rng.random(NUM_SETS) < 0.5
    elements = SETS[solution].sum(axis=0)
    solution_fitness = fitness(solution, elements)
    history = [solution_fitness]
    best_solution = solution.copy()
    best_fitness = solution_fitness
    fitness_calls = 1  # Conta le chiamate alla funzione fitness

    start_time = time.time()

    for step in range(max_steps):
        candidate, elements_candidate = tweak(solution, elements)
        candidate_fitness = fitness(candidate, elements_candidate)
        fitness_calls += 1

        if candidate_fitness > solution_fitness:
            solution = candidate
            elements = elements_candidate
            solution_fitness = candidate_fitness

            if solution_fitness > best_fitness:
                best_solution = solution.copy()
                best_fitness = solution_fitness

        # Se la soluzione copre tutto l'universo, interrompe la ricerca
        if best_fitness[0] == UNIVERSE_SIZE:
            break

    execution_time = time.time() - start_time
    return best_solution, cost(best_solution), step + 1, fitness_calls, execution_time




La funzione `plot_cost_history` permette di visualizzare come il costo cambia nel tempo durante l'esecuzione dell'algoritmo di Hill Climbing. Grazie a questa funzione, possiamo osservare la convergenza e l’efficacia del processo di ottimizzazione.

- **Input**:
  - `cost_history`: una lista di costi registrati a ogni passo dell'algoritmo. Rappresenta l'andamento del costo nel tempo.
  - `instance_idx`: un intero che rappresenta l'indice dell'istanza corrente. Questo serve a etichettare il grafico.

- **Output**: 
  - La funzione non restituisce alcun valore ma genera un grafico che mostra la riduzione del costo in funzione dei passi.

- **Dettagli del Grafico**:
  - L’asse `x` rappresenta i passi dell’algoritmo (iterazioni).
  - L’asse `y` rappresenta il costo associato alla soluzione corrente.
  - Ogni grafico è etichettato con l’indice dell'istanza, utile per confrontare diverse istanze.

In [262]:
def plot_results(history, instance_info):
    """
    Plot dei risultati dell'algoritmo.
    """
    fitness_coverage = [f[0] for f in history]
    fitness_cost = [f[1] for f in history]

    plt.figure(figsize=(14, 8))
    plt.plot(fitness_coverage, label='Copertura')
    plt.plot(fitness_cost, label='Costo')
    plt.title(f"Istanza: {instance_info}")
    plt.xlabel('Passi')
    plt.ylabel('Fitness')
    plt.legend()
    plt.show()



La funzione `analyze_statistics` fornisce una sintesi statistica delle esecuzioni di Hill Climbing su tutte le istanze, riportando informazioni sul costo finale di ciascuna esecuzione. Questa funzione è utile per confrontare le performance e ottenere una visione complessiva dei risultati.

- **Input**:
  - `cost_histories`: una lista di liste. Ogni lista interna rappresenta l'andamento del costo per una singola istanza, registrato a ogni iterazione.
  
- **Output**:
  - Un dizionario con le seguenti statistiche:
    - **Costo Finale Medio**: il costo finale medio su tutte le istanze, utile per capire l’efficacia generale.
    - **Costo Minimo**: il costo più basso ottenuto, indica la miglior performance tra tutte le esecuzioni.
    - **Costo Massimo**: il costo più alto, rappresenta la peggiore performance.
    - **Deviazione Standard**: la variabilità dei costi finali, che può suggerire la stabilità dell'algoritmo.

Questa funzione fornisce un quadro d’insieme delle performance di Hill Climbing, permettendo di interpretare i risultati quantitativamente.

In [263]:
def print_statistics(idx, universe_size, num_sets, density, steps, fitness_calls, total_cost, exec_time):
    """
    Stampa le statistiche per ogni istanza con i dati scritti uno per riga.
    """
    minutes = int(exec_time // 60)
    seconds = int(exec_time % 60)
    time_str = f"{minutes:02d}:{seconds:02d}"

    print(f"ISTANZA {idx}")
    print(f"UNIVERSE SIZE: {universe_size}")
    print(f"NUM SETS: {num_sets}")
    print(f"DENSITY: {density}")
    print(f"STEPS: {steps}")
    print(f"FITNESS CALLS: {fitness_calls}")
    print(f"COST: {total_cost:.6f}")
    print(f"EXECUTION TIME (mm:ss): {time_str}")
    print("-" * 40)


La funzione `main` esegue l'intero flusso di ottimizzazione per il laboratorio sul problema del Set Cover con l'algoritmo di Hill Climbing. I passaggi chiave includono la generazione delle istanze, l’esecuzione dell’algoritmo, la visualizzazione dell'andamento dei costi e un’analisi statistica finale.

1. **Definizione delle Istanze**:
   - Ogni configurazione dell'universo e dei set è specificata in una lista `instances`. Ciascuna istanza è rappresentata da una dimensione dell'universo (`Universe size`), dal numero di set (`Num sets`) e dalla densità (`Density`).
   
2. **Generazione e Configurazione di Istanze**:
   - Per ciascuna istanza, `generate_universe` imposta i parametri specifici e inizializza il generatore di numeri casuali per garantire la riproducibilità.
   - `generate_instance` costruisce la matrice `SETS` e l'array `COSTS` per il problema corrente.

3. **Esecuzione di Hill Climbing**:
   - `hill_climbing` viene chiamato per risolvere il problema del Set Cover per l'istanza corrente. La funzione restituisce la miglior soluzione trovata (`best_solution`), il costo della soluzione (`best_cost`), e un array (`cost_history`) che registra il costo per ciascun passo dell'algoritmo.
   
4. **Visualizzazione dell'Andamento del Costo**:
   - La funzione `plot_cost_history` visualizza l'andamento del costo per ogni istanza. Questo grafico mostra come l'algoritmo cerca di ridurre il costo a ogni iterazione.

5. **Analisi Statistica**:
   - Una volta completate tutte le istanze, `analyze_statistics` raccoglie i dati finali dei costi per calcolare statistiche utili come il costo medio finale, il costo minimo e massimo, e la deviazione standard.
   - I risultati vengono stampati in un riepilogo statistico.

Con questo `main`, abbiamo una panoramica completa delle performance di Hill Climbing su diverse configurazioni del problema del Set Cover, permettendoci di confrontare e interpretare i risultati.

In [264]:
def main():
    instances = [
        (100, 10, 0.2, 100),
        # (1000, 100, 0.2, 1000),
        # (10000, 1000, 0.2, 10000),
        # (100000, 10000, 0.1, 100000),
        # (100000, 10000, 0.2, 100000),
        # (100000, 10000, 0.3, 100000)
    ]

    for idx, (universe_size, num_sets, density, max_steps) in enumerate(instances, 1):
        generate_universe(universe_size, num_sets, density)
        generate_instance()
        best_solution, total_cost, steps, fitness_calls, exec_time = hill_climbing(max_steps)

        minutes = int(exec_time // 60)
        seconds = int(exec_time % 60)
        time_str = f"{minutes:02d}:{seconds:02d}"


if __name__ == "__main__":
    main()

ic| f"Generator initialized for Universe Size: {UNIVERSE_SIZE}, Num Sets: {NUM_SETS}, Density: {DENSITY}": 'Generator initialized for Universe Size: 100, Num Sets: 10, Density: 0.2'
